<a href="https://colab.research.google.com/github/mostofa89/Online_Payments_Fraud_Detection/blob/main/Online_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Online Payment Fraud Detection
   --> Rgression
   --> RandomForest
   --> Neural Network

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score,confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import roc_curve, auc
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression

#Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/AI ML Projeect/online_payments_fraud_detection_dataset (1).csv')

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
print(df.columns)

#Preprocessing

In [ ]:
df["isFraud"].value_counts()

##Cleaning

In [ ]:
df.isnull().sum()

### Drop Unnecessary Columns

In [ ]:
df = df.drop(columns=['nameOrig'])

In [ ]:
df = df.drop(columns=['nameDest'])

In [ ]:
df.columns

### Hanndling Null Values

In [ ]:
med = df['oldbalanceDest'].median()

In [ ]:
med

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
  if df[col].isnull().sum() > 0:
      df[col] = df[col].fillna(df[col].median())



In [ ]:
df.isnull().sum()

In [ ]:
df

### Drop Duplicate Values

In [ ]:
df['errorBalanceOrig'] = df['oldbalanceOrg'] + df['amount'] - df['newbalanceOrig']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df

###Encoding

In [ ]:
onehot = OneHotEncoder(drop='first', sparse_output=False)
type_encoded = onehot.fit_transform(df[['type']])

type_df = pd.DataFrame(type_encoded, columns=onehot.get_feature_names_out(['type']))

df = df.reset_index(drop=True)
type_df = type_df.reset_index(drop=True)

df = pd.concat([df.drop(columns=['type']), type_df], axis=1)

In [ ]:
if 'type' in df.columns:
    df = pd.get_dummies(df, columns=['type'], drop_first=True, dtype=int)
    print("Encoding successful! 'type' column converted to dummy variables.")
else:
    print("'type' column already encoded. Skipping step.")

In [ ]:
df

###Heatmap Represtation

In [ ]:
corr = df.corr()

In [ ]:
plt.figure(figsize=(12, 8))
ax = sns.heatmap(corr, annot=True, cmap='Greens')
plt.title('Correlation Heatmap')
plt.show()

### Split into Train Test

In [ ]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

###Scaling

In [ ]:
continuous_cols = [
  'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
  'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest'
]

In [ ]:
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [ ]:
df

#Model Training

##LogisticRegression

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train, y_train)

In [ ]:
lr_train_pred = lr.predict(X_train)
lr_test_pred = lr.predict(X_test)

In [ ]:
precision = precision_score(y_test, lr_test_pred)
recall = recall_score(y_test, lr_test_pred)
f1 = f1_score(y_test, lr_test_pred)
accuracy = accuracy_score(y_test, lr_test_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

##RandomForest

In [ ]:
X_train_stage2 = X_train.copy()
X_train_stage2['linreg_output'] = lr_train_pred

X_test_stage2 = X_test.copy()
X_test_stage2['linreg_output'] = lr_test_pred

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train_stage2, y_train)

In [ ]:
rf_train_pred = rf.predict(X_train_stage2)
rf_test_pred = rf.predict(X_test_stage2)

In [ ]:
precision = precision_score(y_test, rf_test_pred)
recall = recall_score(y_test, rf_test_pred)
f1 = f1_score(y_test, rf_test_pred)
accuracy = accuracy_score(y_test, rf_test_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

##Neural Network

In [ ]:
rf_train_prob = rf.predict_proba(X_train_stage2)[:, 1]
rf_test_prob = rf.predict_proba(X_test_stage2)[:, 1]

X_train_stage3 = X_train_stage2.copy()
X_train_stage3['rf_output'] = rf_train_prob

X_test_stage3 = X_test_stage2.copy()
X_test_stage3['rf_output'] = rf_test_prob

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu', random_state=42)
mlp.fit(X_train_stage3, y_train)

In [ ]:
mlp_pred = mlp.predict(X_test_stage3)
mlp_prob = mlp.predict_proba(X_test_stage3)[:, 1]

#Model Accuracy Analysis

In [ ]:
y_pred = mlp_pred
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

In [ ]:
print("\nConfusion Matrix:")
conf_mat = confusion_matrix(y_test, y_pred)
print(conf_mat)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

In [ ]:
plt.figure(figsize = (6, 5))
sns.heatmap(conf_mat, annot = True, fmt = 'd', cmap='Blues', xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
plt.title("Confusion Matrix — Hybrid Model", fontsize=14, pad=15)
plt.xlabel("Predicted Label", fontsize=12, labelpad=10)
plt.ylabel("True Label", fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, mlp_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Hybrid Model (AUC = {roc_auc:.3f})', color='darkorange')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Hybrid Model (LR → RF → NN)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"AUC Score: {roc_auc:.4f}")

#Unsupervised Learning

## Clustering

In [ ]:
df_copy = df.copy()
X = df_copy[continuous_cols]

In [ ]:
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [ ]:
k_range = range(1, 10)

SSE = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=1, n_init=10)
    km.fit(X)
    SSE.append(km.inertia_)

SSE

In [ ]:
plt.xlabel("K")
plt.ylabel("Sum of Squarred Error [SSE]")
plt.plot(k_range,SSE)
plt.show()

In [ ]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=1, n_init=10)
kmeans.fit(X_train_cls)

In [ ]:
train_clusters = kmeans.predict(X_train_cls)
test_clusters = kmeans.predict(X_test_cls)

In [ ]:
cluster_vs_fraud = pd.crosstab(train_clusters, y_train, rownames=['Cluster'], colnames=['isFraud'])
print(cluster_vs_fraud)

In [ ]:
centroids = kmeans.cluster_centers_

In [ ]:
cluster_num = 3

color = ['green', 'red', 'blue']

amount_idx = continuous_cols.index('amount')
balance_idx = continuous_cols.index('oldbalanceOrg')

plt.figure(figsize=(10, 7))

for i in range(cluster_num):

  plt.scatter(
    X_train_cls.iloc[train_clusters == i, amount_idx],
    X_train_cls.iloc[train_clusters == i, balance_idx],
    color=color[i],
    alpha=0.5,
    label=f'Cluster {i+1}'
    )

  plt.scatter(
    kmeans.cluster_centers_[i, amount_idx],
    kmeans.cluster_centers_[i, balance_idx],
    color=color[i],
    s=250,
    marker='X',
    edgecolor='black',
    linewidth=2,
    label=f'Centroid {i+1}'
    )

plt.xlabel('Amount')
plt.ylabel('Old Balance Origin')
plt.title('K-Means Clustering with Centroids')

plt.legend()
plt.grid(True)

plt.show()